# Photos Library Duplicate Cleanup Notebook

Report-only v1. This notebook does **not** delete anything.

Run order:
1. Run configuration.
2. Load/build inventory.
3. Fill identity fields.
4. Group and analyze duplicate candidates.
5. Write permanent operation report.

Helper functions live in `photos_duplicate_cleanup_helpers.py` so the notebook stays readable.


In [1]:
# ============================================================
# Cell 1. Configuration
# ============================================================

from pathlib import Path
import os
import sys
import json
from datetime import datetime

PROJECT_ROOT = Path("/Users/huohsien/workspace/python/explore_photos_library")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LIBRARY_ID = "current_default"
LIBRARY_PATH = Path("/Users/huohsien/Pictures/Photos Library.photoslibrary")

CACHE_DIR = PROJECT_ROOT / "data" / "inventory_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_NAME = LIBRARY_ID
INVENTORY_CACHE_PATH = CACHE_DIR / f"{CACHE_NAME}.inventory.pkl.gz"

REPORTS_ROOT = PROJECT_ROOT / "IMPORTANT_Photos_Library_Critical_Operation_Reports"
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
REPORT_DIR = REPORTS_ROOT / f"{RUN_TIMESTAMP}__Photos_Library_Duplicate_Cleanup__{LIBRARY_ID}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("LIBRARY_ID:", LIBRARY_ID)
print("LIBRARY_PATH:", LIBRARY_PATH)
print("INVENTORY_CACHE_PATH:", INVENTORY_CACHE_PATH)
print("REPORT_DIR:", REPORT_DIR)


LIBRARY_ID: current_default
LIBRARY_PATH: /Users/huohsien/Pictures/Photos Library.photoslibrary
INVENTORY_CACHE_PATH: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/current_default.inventory.pkl.gz
REPORT_DIR: /Users/huohsien/workspace/python/explore_photos_library/IMPORTANT_Photos_Library_Critical_Operation_Reports/20260607-163121__Photos_Library_Duplicate_Cleanup__current_default


In [2]:
# ============================================================
# Cell 2. Load or build inventory
# ============================================================

import osxphotos
from photos_inventory import (
    build_inventory,
    print_inventory_summary,
    save_inventory_cache,
    load_inventory_cache,
)

FORCE_REBUILD_INVENTORY = False

if INVENTORY_CACHE_PATH.exists() and not FORCE_REBUILD_INVENTORY:
    inventory = load_inventory_cache(
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )
    print("Loaded inventory cache:", INVENTORY_CACHE_PATH)
else:
    print("Building inventory from Photos Library:")
    print(LIBRARY_PATH)

    photosdb = osxphotos.PhotosDB(dbfile=str(LIBRARY_PATH))
    osx_assets = photosdb.photos()

    inventory = build_inventory(osx_assets)

    save_inventory_cache(
        inventory=inventory,
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )

    print("Saved inventory cache:", INVENTORY_CACHE_PATH)

print_inventory_summary(inventory)


loaded inventory cache: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/current_default.inventory.pkl.gz
elapsed seconds: 0.22
inventory assets: 94678
inventory albums: 5952
inventory folders: 33
movies: 7406
hidden: 0
favorites: 768
descriptions: 2276
keywords: 27529
Loaded inventory cache: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/current_default.inventory.pkl.gz
inventory assets: 94678
inventory albums: 5952
inventory folders: 33
movies: 7406
hidden: 0
favorites: 768
descriptions: 2276
keywords: 27529


In [3]:
# ============================================================
# Cell 3. Fill identity fields
# ============================================================

from photos_duplicate_cleanup_helpers import fill_duplicate_cleanup_identity_fields

fill_duplicate_cleanup_identity_fields(inventory)


Filled identity fields
asset count: 94678
base_id filled: 94639
unique_id filled: 94639

base_id failure reasons:
- missing file_size_bytes: 39

First 3 assets after fill:
--------------------------------------------------------------------------------
original_filename: IMG_1338.PNG
date: 2025-09-07T07:22:31.194874+08:00
path: /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/F/FCA6CAB0-8709-405E-AA8C-7DD2DAB3827C.png
file_size_bytes: 1440561
base_id: ('IMG_1338.PNG', '09-07 07:22:31.194874', 1440561)
unique_id: (('IMG_1338.PNG', '09-07 07:22:31.194874', 1440561), (('description', None), ('keywords', ('HIDE', 'NSFW', 'NSFW_TUMBLR')), ('favorite', False), ('hidden', False), ('album_titles', ('Tumblr Girls - 1 (for fixing albums that disappeared after Apple iCloud Crash 2025 Mar. 25, including adding new images and videos downloaded after the tragedy)',)), ('folder_paths', ())))
--------------------------------------------------------------------------------
original_filen

In [4]:
# ============================================================
# Cell 4. Group duplicate candidates
# ============================================================

from photos_duplicate_cleanup_helpers import group_assets_by_field

unique_id_groups, assets_without_unique_id = group_assets_by_field(
    inventory,
    "photo_library_asset_unique_id",
)

duplicate_candidate_groups = {
    unique_id: group
    for unique_id, group in unique_id_groups.items()
    if len(group) > 1
}

print("assets:", len(inventory["assets"]))
print("generated unique_id count:", len(unique_id_groups))
print("assets without unique_id:", len(assets_without_unique_id))
print("duplicate candidate group count:", len(duplicate_candidate_groups))
print("duplicate candidate asset count:", sum(len(group) for group in duplicate_candidate_groups.values()))

if assets_without_unique_id:
    print()
    print("First assets without unique_id:")
    for asset in assets_without_unique_id[:10]:
        print(
            asset.get("original_filename"),
            asset.get("uuid"),
            asset.get("asset_scope"),
            asset.get("path"),
        )


assets: 94678
generated unique_id count: 94616
assets without unique_id: 39
duplicate candidate group count: 23
duplicate candidate asset count: 46

First assets without unique_id:
IMG_1323.JPG 73B8C09C-6F12-4A25-9616-342376038717 PATH_NONE None
IMG_5763.JPG 4DE92713-D352-48C8-85CB-0A73D6D068EE PATH_NONE None
IMG_1334.jpeg 2E53DF99-1D66-4B60-AF82-382165581532 PATH_NONE None
IMG_0135.jpeg F64DC6F0-F5C9-4C42-BF7A-765DEC3EC2B9 PATH_NONE None
IMG_6723.JPG AE1A88BB-EEDD-4444-ADCC-0888DC07F234 PATH_NONE None
IMG_5986.jpeg 409C7D86-70CE-4CB5-BF9F-400FF703F14A PATH_NONE None
IMG_2037.JPG 3F72A7D2-4A62-4647-BE91-1DFFF759BD41 PATH_NONE None
IMG_1765.jpeg CB5BD81F-4844-411B-B1C0-67A2BA0CC5C7 PATH_NONE None
IMG_2054.MOV CE2621A6-2BBB-4772-AC6E-14A66B86CA37 PATH_NONE None
IMG_1354.JPG 771B044C-F2CE-4265-AAB1-F7597A941549 PATH_NONE None


In [5]:
# ============================================================
# Cell 5. Analyze duplicate candidates
# ============================================================

from photos_duplicate_cleanup_helpers import analyze_duplicate_candidate_groups

duplicate_analysis = analyze_duplicate_candidate_groups(duplicate_candidate_groups)


Analyzing group 1/23
Analyzing group 10/23
Analyzing group 20/23
Analyzing group 23/23
analysis group count: 23
elapsed seconds: 35.215


In [6]:
# ============================================================
# Cell 6. Summary
# ============================================================

from photos_duplicate_cleanup_helpers import count_records_by_status

status_counts = count_records_by_status(duplicate_analysis)

delete_candidate_count = sum(
    len(record.get("delete_candidates") or [])
    for record in duplicate_analysis
)

print("status counts:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

print("delete candidate asset count:", delete_candidate_count)

print()
print("First deletable duplicate groups:")
printed = 0

for record in duplicate_analysis:
    if record.get("status") != "DELETABLE_DUPLICATE":
        continue

    print("-" * 80)
    print("reason:", record.get("reason"))
    print("asset_count:", record.get("asset_count"))
    print("keep_assets:", len(record.get("keep_assets") or []))
    print("delete_candidates:", len(record.get("delete_candidates") or []))

    for asset in (record.get("keep_assets") or []):
        print("  KEEP:", asset["original_filename"], asset["date_added"], asset["path"])

    for asset in (record.get("delete_candidates") or []):
        print("  DELETE:", asset["original_filename"], asset["date_added"], asset["path"])

    printed += 1

    if printed >= 10:
        print("... more groups not printed")
        break


status counts:
  DELETABLE_DUPLICATE: 23
delete candidate asset count: 23

First deletable duplicate groups:
--------------------------------------------------------------------------------
reason: SAME_UNIQUE_ID_AND_SAME_SHA256
asset_count: 2
keep_assets: 1
delete_candidates: 1
  KEEP: IMG_4982.MOV 2022-06-30T11:10:05.050800+08:00 /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/4/4C6C5B1E-B0AF-4708-89C2-5CDA30A59497.mov
  DELETE: IMG_4982.MOV 2023-11-05T16:55:08.727065+08:00 /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/8/88ACF87B-2767-4200-AF08-81F9AD569C5D.mov
--------------------------------------------------------------------------------
reason: SAME_UNIQUE_ID_AND_SAME_SHA256
asset_count: 2
keep_assets: 1
delete_candidates: 1
  KEEP: IMG_0106.JPG 2021-12-15T19:26:49.506998+08:00 /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/A/AC408AAA-DD22-4340-B4B9-B9CE2D104627.jpeg
  DELETE: IMG_0106.JPG 2021-12-15T21:51:40.174364+08:00 /Users/

In [7]:
# ============================================================
# Cell 7. Write permanent operation report
# ============================================================

from photos_duplicate_cleanup_helpers import write_operation_report

report_result = write_operation_report(
    report_dir=REPORT_DIR,
    duplicate_analysis=duplicate_analysis,
    inventory=inventory,
    assets_without_unique_id=assets_without_unique_id,
    duplicate_candidate_groups=duplicate_candidate_groups,
    run_timestamp=RUN_TIMESTAMP,
    library_id=LIBRARY_ID,
    library_path=LIBRARY_PATH,
    inventory_cache_path=INVENTORY_CACHE_PATH,
)

delete_candidate_rows = report_result["delete_candidate_rows"]
keep_asset_rows = report_result["keep_asset_rows"]
location_conflict_rows = report_result["location_conflict_rows"]


Photos Library Duplicate Cleanup Report

run_timestamp: 20260607-163121
library_id: current_default
library_path: /Users/huohsien/Pictures/Photos Library.photoslibrary
inventory_cache_path: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/current_default.inventory.pkl.gz

asset_count: 94678
assets_without_unique_id: 39
duplicate_candidate_group_count: 23
duplicate_candidate_asset_count: 46

status_counts:
{
  "DELETABLE_DUPLICATE": 23
}

delete_candidate_asset_count: 23
keep_asset_count: 23
location_conflict_row_count: 0

IMPORTANT:
This report folder should be copied/backed up to Google Drive:
/Users/huohsien/workspace/python/explore_photos_library/IMPORTANT_Photos_Library_Critical_Operation_Reports/20260607-163121__Photos_Library_Duplicate_Cleanup__current_default

Report files:
  /Users/huohsien/workspace/python/explore_photos_library/IMPORTANT_Photos_Library_Critical_Operation_Reports/20260607-163121__Photos_Library_Duplicate_Cleanup__current_default/REA

In [8]:
# ============================================================
# Cell 8. Optional: print manual deletion list
# ============================================================
#
# This notebook does NOT delete anything from Photos Library.
# It only produces a report and delete candidate list.
#
# For actual deletion, review delete_candidates.tsv first.

for row in delete_candidate_rows[:100]:
    print(
        row["original_filename"],
        row["date"],
        row["date_added"],
        row["path"],
    )

if len(delete_candidate_rows) > 100:
    print("... more delete candidates not printed")


IMG_4982.MOV 2022-06-30T11:05:36+08:00 2023-11-05T16:55:08.727065+08:00 /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/8/88ACF87B-2767-4200-AF08-81F9AD569C5D.mov
IMG_0106.JPG 2021-12-15T19:26:48.808220+08:00 2021-12-15T21:51:40.174364+08:00 /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/1/14A8F1D8-2906-4068-8760-E89903F2AE58.jpeg
IMG_2344.PNG 2017-09-29T21:22:02+08:00 2017-10-13T13:28:35.838223+08:00 /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/0/09E67C6F-DADE-4EA0-BDD7-8431A2A973A4.png
IMG_4980.MOV 2022-06-30T10:43:45+08:00 2023-11-05T16:54:59.163527+08:00 /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/F/F9496917-1F03-4B35-B608-9E9EBA9E688E.mov
IMG_0234.PNG 2020-11-19T00:23:16.639491+08:00 2020-11-19T00:23:29.247762+08:00 /Users/huohsien/Pictures/Photos Library.photoslibrary/originals/9/956623AF-8FEF-4032-948E-816E52B2EE6F.png
3320474716416102425.mp4 2020-07-15T17:39:24+08:00 2020-07-16T12:52:03.116081+08:00 /Users/